In [2]:
# [IA] Gemini Extração de métricas estruturais (ordem, tamanho, densidade, graus e pesos) da lista_nos para DataFrames pandas
import sys
import os
import pandas as pd

# Garante que o Python enxergue a pasta src/ a partir de notebooks/
sys.path.append(os.path.abspath(".."))

from src.graphs.io import carregar_grafo_tmdb

# 1. Carrega o grafo usando o io.py já implementado
lista_nos, mapa_nos = carregar_grafo_tmdb("../data/dataset_parte2/")

# 2. Métricas globais: Ordem |V|, Tamanho |E| e Densidade (grafo dirigido)
V = len(lista_nos)
E = sum(len(no.lista_arestas) for no in lista_nos)
densidade = E / (V * (V - 1)) if V >= 2 else 0.0

print(f"Ordem |V|: {V}")
print(f"Tamanho |E|: {E}")
print(f"Densidade: {densidade:.8f}")

# 3. Extração de graus de entrada, graus de saída e lista de arestas
grau_saida = {no.id: len(no.lista_arestas) for no in lista_nos}
grau_entrada = {no.id: 0 for no in lista_nos}
registros_arestas = []

for no in lista_nos:
    for aresta in no.lista_arestas:
        grau_entrada[aresta.destino.id] += 1
        registros_arestas.append({
            "origem_id": no.id,
            "origem_titulo": no.titulo,
            "destino_id": aresta.destino.id,
            "destino_titulo": aresta.destino.titulo,
            "peso": aresta.custo
        })

# DataFrame de vértices (para analisar hubs e distribuição de graus)
df_nos = pd.DataFrame({
    "id": [no.id for no in lista_nos],
    "titulo": [no.titulo for no in lista_nos],
    "grau_entrada": [grau_entrada[no.id] for no in lista_nos],
    "grau_saida": [grau_saida[no.id] for no in lista_nos],
})
df_nos["grau_total"] = df_nos["grau_entrada"] + df_nos["grau_saida"]

# DataFrame de arestas (para analisar distribuição de pesos negativos/positivos/zero)
df_arestas = pd.DataFrame(registros_arestas)
df_arestas["sinal_peso"] = pd.cut(
    df_arestas["peso"],
    bins=[-float("inf"), -1e-9, 1e-9, float("inf")],
    labels=["Negativo (< 0)", "Zero (= 0)", "Positivo (> 0)"]
)

display(df_nos.sort_values("grau_total", ascending=False).head(10))
print("\nContagem por sinal do peso:")
print(df_arestas["sinal_peso"].value_counts())

Ordem |V|: 416243
Tamanho |E|: 500000
Densidade: 0.00000289


,id,titulo,grau_entrada,grau_saida,grau_total
1132,194055,"1776, or The Hessian Renegades",173,836,1009
1371,193964,The Usurer,241,725,966
1106,384913,A Strange Meeting,135,820,955
1274,487462,The Newlyweds,197,750,947
1802,522108,The Long Road,204,726,930
1693,334061,The Primal Call,168,740,908
1696,193953,Fighting Blood,216,683,899
1308,418136,The Two Brothers,213,677,890
1395,194000,The Oath and the Man,242,636,878
1137,387015,The Broken Locket,156,722,878



Contagem por sinal do peso:
sinal_peso
Positivo (> 0)    292659
Negativo (< 0)    190441
Zero (= 0)         16900
Name: count, dtype: int64


In [3]:
# [IA] Gemini Cruzamento de arestas com o elenco compartilhado e atributos numéricos para análise multivariada
import pandas as pd

# 1. Carrega o dataset limpo com as colunas de elenco e atributos numéricos/categóricos
# (Ajuste o caminho se estiver lendo o CSV original completo para pegar budget, runtime, popularity, genres)
df_filmes = pd.read_csv("../data/dataset_parte2/tmdb_limpo.csv.gz")
df_filmes["id"] = df_filmes["id"].astype(str)
df_filmes["ano"] = pd.to_datetime(df_filmes["release_date"], errors="coerce").dt.year

# Mapeia cada filme para o conjunto (set) de seus atores
elenco_por_filme = {
    row.id: {a.strip() for a in str(row.cast).split(",") if a.strip()}
    for row in df_filmes[["id", "cast"]].itertuples(index=False)
}

# 2. Identifica os atores em comum entre origem e destino de cada aresta
atores_compartilhados = [
    list(elenco_por_filme.get(orig, set()) & elenco_por_filme.get(dest, set()))
    for orig, dest in zip(df_arestas["origem_id"], df_arestas["destino_id"])
]

df_arestas_atores = df_arestas.copy()
df_arestas_atores["atores_comuns"] = atores_compartilhados

# Explode a lista para ter uma linha por (ator, transição de filme)
df_por_ator = df_arestas_atores.explode("atores_comuns").rename(columns={"atores_comuns": "ator"})
df_por_ator = df_por_ator.dropna(subset=["ator"])

# 3. Agregação por ator (média da variação de nota, soma dos pesos negativos e total de transições)
resumo_atores = (
    df_por_ator.groupby("ator", as_index=False)
    .agg(
        total_transicoes=("peso", "count"),
        media_variacao_nota=("peso", "mean"),
        soma_variacao_nota=("peso", "sum"),
        qtd_quedas=("peso", lambda s: (s < 0).sum()),
    )
)
resumo_atores["proporcao_quedas"] = resumo_atores["qtd_quedas"] / resumo_atores["total_transicoes"]

display(resumo_atores.sort_values("soma_variacao_nota").head(15))

,ator,total_transicoes,media_variacao_nota,soma_variacao_nota,qtd_quedas,proporcao_quedas
1682,Chuck Connors,3157,-0.549889,-1736.0,2028,0.642382
3606,Georges Paulais,4277,-0.173977,-744.1,2317,0.541735
2412,Eduard von Winterstein,4851,-0.139435,-676.4,2436,0.502165
6912,Marion Leonard,5456,-0.079857,-435.7,2661,0.487720
2776,Ernst Behmer,2145,-0.177809,-381.4,1023,0.476923
1925,Daniel Mendaille,1325,-0.239472,-317.3,711,0.536604
5542,John R. Cumpson,1478,-0.207375,-306.5,872,0.589986
4068,Harald Madsen,781,-0.335339,-261.9,446,0.571063
6501,Lucien Callamand,3915,-0.057880,-226.6,1973,0.503959
4342,Henri Vilbert,2628,-0.075038,-197.2,1311,0.498858


In [4]:
# [IA] Gemini Merge com atributos do filme de destino e cálculo de matriz de correlação
colunas_extra = [c for c in ["id", "ano", "imdb_rating", "budget", "revenue", "popularity", "runtime"] if c in df_filmes.columns]

df_multivariado = df_arestas.merge(
    df_filmes[colunas_extra],
    left_on="destino_id",
    right_on="id",
    how="left"
)

# Calcula coeficientes de correlação linear (Pearson) e monotônica (Spearman)
corr_pearson = df_multivariado.drop(columns=["origem_id", "origem_titulo", "destino_id", "destino_titulo", "id", "sinal_peso"], errors="ignore").corr(method="pearson")
corr_spearman = df_multivariado.drop(columns=["origem_id", "origem_titulo", "destino_id", "destino_titulo", "id", "sinal_peso"], errors="ignore").corr(method="spearman")

print("Correlação de Pearson com o peso (variação de nota):")
print(corr_pearson["peso"].sort_values())

print("\nCorrelação de Spearman com o peso (variação de nota):")
print(corr_spearman["peso"].sort_values())

Correlação de Pearson com o peso (variação de nota):
ano            0.034871
imdb_rating    0.652148
peso           1.000000
Name: peso, dtype: float64

Correlação de Spearman com o peso (variação de nota):
ano            0.012559
imdb_rating    0.618118
peso           1.000000
Name: peso, dtype: float64


## Gráficos de AVD: atores, receita, idioma, produtoras e graus

In [ ]:
# [IA] Claude Code – carrega o dataset limpo e junta revenue, idioma e produtoras do dataset bruto (pelo id)
import os
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MIN_FILMES_VARIACAO = 10    # mínimo de filmes para entrar no ranking de variação
MIN_FILMES_CONSTANCIA = 20  # mínimo de filmes para medir o desvio padrão
MIN_FILMES_COM_RECEITA = 3  # mínimo de filmes com revenue > 0 por ator
TOP_N = 15

pasta_bruto = kagglehub.dataset_download("alanvourch/tmdb-movies-daily-updates")
bruto = pd.read_csv(
    os.path.join(pasta_bruto, "TMDB_all_movies.csv"),
    usecols=["id", "revenue", "original_language", "production_companies"],
    low_memory=False,
).drop_duplicates("id")

filmes = pd.read_csv("../data/dataset_parte2/tmdb_limpo.csv.gz")
filmes["release_date"] = pd.to_datetime(filmes["release_date"], errors="coerce")
filmes = filmes.dropna(subset=["release_date", "imdb_rating", "cast"]).drop_duplicates("id")
filmes = filmes.merge(bruto, on="id", how="left")
filmes.shape

In [ ]:
# [IA] Claude Code – tabela (ator, filme) em ordem cronológica e variação de nota por ator
longo = filmes[["id", "release_date", "imdb_rating", "revenue"]].assign(ator=filmes["cast"].str.split(","))
longo = longo.explode("ator")
longo["ator"] = longo["ator"].str.strip()
longo = longo[longo["ator"] != ""].drop_duplicates(["ator", "id"])
longo = longo.sort_values(["ator", "release_date", "id"]).reset_index(drop=True)

# Variação = nota média da 2ª metade da carreira - nota média da 1ª metade (filmes ordenados por data)
posicao = longo.groupby("ator").cumcount()
total = longo.groupby("ator")["id"].transform("size")
metade = total // 2
primeira = longo[posicao < metade]
segunda = longo[posicao >= total - metade]
variacao = pd.DataFrame({
    "nota_inicio": primeira.groupby("ator")["imdb_rating"].mean(),
    "nota_fim": segunda.groupby("ator")["imdb_rating"].mean(),
    "filmes": longo.groupby("ator")["id"].size(),
})
variacao = variacao[variacao["filmes"] >= MIN_FILMES_VARIACAO].copy()
variacao["variacao"] = variacao["nota_fim"] - variacao["nota_inicio"]
variacao = variacao.sort_values("variacao")
variacao.head()

In [ ]:
# [IA] Claude Code – gráfico de barras horizontais: atores cujas notas mais diminuíram
destaque = variacao.head(TOP_N).sort_values("variacao", ascending=True)
fig, ax = plt.subplots(figsize=(10, 7))
rotulos = [f"{ator} ({int(f)} filmes)" for ator, f in zip(destaque.index, destaque["filmes"])]
ax.barh(rotulos, destaque["variacao"], color="#c92a2a")
ax.invert_yaxis()
ax.set_xlabel("Variação da nota média (2ª metade - 1ª metade da carreira)")
ax.set_title(f"Atores cujas notas mais diminuíram (mín. {MIN_FILMES_VARIACAO} filmes)")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# [IA] Claude Code – gráfico de barras horizontais: atores cujas notas mais aumentaram
destaque = variacao.tail(TOP_N).sort_values("variacao", ascending=False)
fig, ax = plt.subplots(figsize=(10, 7))
rotulos = [f"{ator} ({int(f)} filmes)" for ator, f in zip(destaque.index, destaque["filmes"])]
ax.barh(rotulos, destaque["variacao"], color="#2b8a3e")
ax.invert_yaxis()
ax.set_xlabel("Variação da nota média (2ª metade - 1ª metade da carreira)")
ax.set_title(f"Atores cujas notas mais aumentaram (mín. {MIN_FILMES_VARIACAO} filmes)")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# [IA] Claude Code – boxplot dos atores mais constantes (menor desvio padrão das notas)
resumo = longo.groupby("ator")["imdb_rating"].agg(["std", "size"])
resumo = resumo[resumo["size"] >= MIN_FILMES_CONSTANCIA].sort_values("std").head(TOP_N)
dados = [longo.loc[longo["ator"] == ator, "imdb_rating"] for ator in resumo.index]
rotulos = [f"{ator}\n(dp {std:.2f}, n={n})" for ator, std, n in zip(resumo.index, resumo["std"], resumo["size"])]

fig, ax = plt.subplots(figsize=(10, 8))
ax.boxplot(dados, tick_labels=rotulos, orientation="horizontal", patch_artist=True,
           boxprops={"facecolor": "#a5d8ff"}, medianprops={"color": "#1c1c1c"})
ax.invert_yaxis()
ax.set_xlabel("imdb_rating dos filmes")
ax.set_title(f"Atores mais constantes: menor desvio padrão das notas (mín. {MIN_FILMES_CONSTANCIA} filmes)")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# [IA] Claude Code – dispersão: variação da nota do ator x receita mediana dos filmes, com linha de tendência
com_receita = longo[longo["revenue"] > 0]
receita = com_receita.groupby("ator")["revenue"].agg(receita_mediana="median", filmes_com_receita="size")
dados = variacao.join(receita, how="inner")
dados = dados[dados["filmes_com_receita"] >= MIN_FILMES_COM_RECEITA]

x = dados["variacao"]
y = np.log10(dados["receita_mediana"])
spearman = dados["variacao"].corr(dados["receita_mediana"], method="spearman")
angular, linear = np.polyfit(x, y, 1)

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(x, y, s=5, alpha=0.15, color="#4c6ef5", linewidths=0, label="Atores")
maiores = dados.nlargest(TOP_N, "variacao")
ax.scatter(maiores["variacao"], np.log10(maiores["receita_mediana"]), s=34, color="#2b8a3e",
           label=f"{TOP_N} que mais aumentaram")
xs = np.linspace(x.min(), x.max(), 100)
ax.plot(xs, angular * xs + linear, color="#e8590c", linewidth=2, label="Linha de tendência")
ax.set_xlabel("Variação da nota média do ator (2ª metade - 1ª metade da carreira)")
ax.set_ylabel("Receita mediana dos filmes do ator (log10 US$)")
ax.set_title(f"Variação das notas x receita dos filmes (Spearman = {spearman:.2f}, n = {len(dados)} atores)")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# [IA] Claude Code – boxplot do imdb_rating por idioma original (12 idiomas com mais filmes)
base = filmes.dropna(subset=["original_language"])
contagem = base["original_language"].value_counts()
idiomas = contagem.head(12).index
ordem = base[base["original_language"].isin(idiomas)].groupby("original_language")["imdb_rating"].median().sort_values(ascending=False).index
dados = [base.loc[base["original_language"] == i, "imdb_rating"] for i in ordem]
rotulos = [f"{i}\n(n={contagem[i]})" for i in ordem]

fig, ax = plt.subplots(figsize=(13, 7))
ax.boxplot(dados, tick_labels=rotulos, patch_artist=True, flierprops={"markersize": 2, "alpha": 0.3},
           boxprops={"facecolor": "#ffd8a8"}, medianprops={"color": "#1c1c1c"})
ax.set_xlabel("original_language")
ax.set_ylabel("imdb_rating")
ax.set_title("imdb_rating por idioma original (os 12 idiomas com mais filmes)")
ax.grid(axis="y", alpha=0.25)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# [IA] Claude Code – boxplot do imdb_rating por produtora (15 com mais filmes, mínimo de 100 filmes)
prod = filmes.dropna(subset=["production_companies"])[["imdb_rating", "production_companies"]].copy()
prod["produtora"] = prod["production_companies"].str.split(",")
prod = prod.explode("produtora")
prod["produtora"] = prod["produtora"].str.strip()
prod = prod[prod["produtora"] != ""]

contagem = prod["produtora"].value_counts()
top = contagem[contagem >= 100].head(15).index
ordem = prod[prod["produtora"].isin(top)].groupby("produtora")["imdb_rating"].median().sort_values(ascending=False).index
dados = [prod.loc[prod["produtora"] == p, "imdb_rating"] for p in ordem]
rotulos = [f"{p}\n(n={contagem[p]})" for p in ordem]

fig, ax = plt.subplots(figsize=(13, 7))
ax.boxplot(dados, tick_labels=rotulos, patch_artist=True, flierprops={"markersize": 2, "alpha": 0.3},
           boxprops={"facecolor": "#ffd8a8"}, medianprops={"color": "#1c1c1c"})
ax.set_xlabel("production_companies")
ax.set_ylabel("imdb_rating")
ax.set_title("imdb_rating por produtora (as 15 com mais filmes)")
ax.grid(axis="y", alpha=0.25)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# [IA] Claude Code – histograma do grau de entrada (usa df_nos da primeira célula; eixo y em escala log)
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df_nos["grau_entrada"], bins=range(0, df_nos["grau_entrada"].max() + 2), color="#4c6ef5")
ax.set_yscale("log")
ax.set_xlabel("Grau de entrada")
ax.set_ylabel("Número de vértices (escala log)")
ax.set_title(f"Distribuição do grau de entrada ({len(df_nos)} vértices, máx. {df_nos['grau_entrada'].max()})")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# [IA] Claude Code – histograma do grau de saída (usa df_nos da primeira célula; eixo y em escala log)
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df_nos["grau_saida"], bins=range(0, df_nos["grau_saida"].max() + 2), color="#e8590c")
ax.set_yscale("log")
ax.set_xlabel("Grau de saída")
ax.set_ylabel("Número de vértices (escala log)")
ax.set_title(f"Distribuição do grau de saída ({len(df_nos)} vértices, máx. {df_nos['grau_saida'].max()})")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()